# Phase B Step 7: Stem Curve Extraction (CVXPY) — All 65 Available Mixes

CPU notebook. Downloads stems from HF, runs CVXPY optimizer, uploads curves.
Run with Accelerator = None (CPU). Resumable — skips completed mixes automatically.

**Targets ALL 65 mixes** with residuals on HF (not just manifest subset).

**DRY_RUN:** Set `True` to process only 1 mix (quick error check).

**Prerequisite:** Step 6 must be complete for the mixes (stems + residuals on HF).

In [ ]:
!pip install -q huggingface_hub soundfile librosa cvxpy ecos clarabel

import subprocess, sys
result = subprocess.run(
    ['git', 'clone', '--depth', '1', 'https://github.com/Uday-461/ai-dj-v4.git', '/kaggle/working/ai-dj/v4'],
    capture_output=True, text=True
)
if result.returncode != 0 and 'already exists' not in result.stderr:
    print('Clone failed:', result.stderr)
else:
    print('Repo ready at /kaggle/working/ai-dj/v4')

!pip install -q -e /kaggle/working/ai-dj/v4
sys.path.insert(0, '/kaggle/working/ai-dj/v4')

In [ ]:
import os

HF_TOKEN      = "hf_..."                   # <-- paste your HuggingFace token here
HF_REPO       = "Uday-4/djmix-v3"
DATA_ROOT     = "/kaggle/working/djmix"
CURVE_WORKERS = 4                           # 28GB RAM: can handle 4 workers
DRY_RUN       = False                       # Set True to process only 1 mix

# -------------------------------------------------------
os.makedirs(DATA_ROOT, exist_ok=True)
os.environ["AIDJ_DATA_ROOT"] = DATA_ROOT

if HF_TOKEN.startswith("hf_..."):
    raise ValueError("Edit Cell 2: set HF_TOKEN to your real HuggingFace token")

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
try:
    api.whoami()
    print(f'HF auth OK — repo: {HF_REPO}')
except Exception as e:
    raise RuntimeError(f'HF auth failed: {e}')

In [ ]:
import json
import pickle
import shutil
import tempfile
from pathlib import Path
from huggingface_hub import hf_hub_download, list_repo_files
from collections import defaultdict

DATA_ROOT_PATH = Path(DATA_ROOT)
PROGRESS_KEY = "phase_b_progress_step7_all65.json"

print(f'Progress file: {PROGRESS_KEY}')

def load_progress():
    """Download progress file from HF (or start fresh)."""
    try:
        local = hf_hub_download(
            repo_id=HF_REPO, filename=PROGRESS_KEY,
            repo_type="dataset", token=HF_TOKEN,
            local_dir=DATA_ROOT, force_download=True,
        )
        with open(local) as f:
            p = json.load(f)
        print(f'Loaded progress: {PROGRESS_KEY}')
        return p
    except Exception:
        print('No progress file on HF — starting fresh')
        return {"curves": []}

def save_progress_local(progress):
    path = DATA_ROOT_PATH / PROGRESS_KEY
    with open(path, 'w') as f:
        json.dump(progress, f, indent=2)
    return path

def push_progress(progress):
    """Upload progress file to HF with retry logic."""
    local_path = save_progress_local(progress)
    max_retries = 3
    for attempt in range(1, max_retries + 1):
        try:
            api.upload_file(
                path_or_fileobj=str(local_path),
                path_in_repo=PROGRESS_KEY,
                repo_id=HF_REPO,
                repo_type="dataset",
                token=HF_TOKEN,
                commit_message="Step 7 progress (all65)",
            )
            return
        except Exception as e:
            if attempt < max_retries:
                print(f'  Upload attempt {attempt} failed, retrying...')
                time.sleep(5)
            else:
                print(f'  Upload failed after {max_retries} attempts: {e}')

# --- Build mix list from ALL mixes with residuals on HF ---
def build_mix_list_from_residuals():
    """Find all mixes with residuals on HF, download transition PKLs, build mix list."""
    print('\nScanning HF for mixes with residuals...')
    all_hf_files = list(list_repo_files(repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN))
    
    # Find all mixes with residuals
    residual_files = [f for f in all_hf_files if f.startswith("results/residuals/") and f.endswith(".npz")]
    mixes_with_residuals = set()
    for f in residual_files:
        parts = f.split("/")
        mix_id = parts[2]
        mixes_with_residuals.add(mix_id)
    
    print(f'Found {len(mixes_with_residuals)} mixes with residuals on HF')
    
    # Download transition PKLs
    trans_dir = DATA_ROOT_PATH / "results" / "transitions"
    trans_dir.mkdir(parents=True, exist_ok=True)
    
    for mix_id in sorted(mixes_with_residuals):
        local = trans_dir / f"{mix_id}.pkl"
        if not local.exists():
            try:
                hf_hub_download(
                    repo_id=HF_REPO, filename=f"results/transitions/{mix_id}.pkl",
                    repo_type="dataset", token=HF_TOKEN,
                    local_dir=DATA_ROOT,
                )
            except Exception as e:
                print(f'Warning: could not download transitions for {mix_id}: {e}')
    
    # Build mix list
    mixes = []
    for mix_id in sorted(mixes_with_residuals):
        pkl_path = trans_dir / f"{mix_id}.pkl"
        if not pkl_path.exists():
            print(f'  {mix_id}: no transition PKL, skipping')
            continue
        with open(pkl_path, 'rb') as f:
            transitions = pickle.load(f)
        track_ids = set()
        for t in transitions:
            if t.get("track_id_prev"): track_ids.add(t["track_id_prev"])
            if t.get("track_id_next"): track_ids.add(t["track_id_next"])
        mixes.append({
            "id": mix_id,
            "track_ids": sorted(track_ids),
            "transitions": transitions,
            "n_transitions": len(transitions),
        })
    
    return mixes

# --- Run ---
import time
progress = load_progress()
all_mixes = build_mix_list_from_residuals()

# Load all progress from Step 7 (across all notebooks)
all_hf_files = list(list_repo_files(repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN))
prog_files = [f for f in all_hf_files if f.startswith("phase_b_progress_step7_") and f.endswith(".json")]
step7_done = set()
for pf in prog_files:
    try:
        local = hf_hub_download(
            repo_id=HF_REPO, filename=pf,
            repo_type="dataset", token=HF_TOKEN,
            local_dir=DATA_ROOT, force_download=True,
        )
        with open(local) as f:
            p = json.load(f)
        step7_done.update(p.get("curves", []))
    except Exception as e:
        print(f'Warning: could not load {pf}: {e}')

# Pending = mixes not yet done
pending_mixes = [m for m in all_mixes if m["id"] not in step7_done]

if DRY_RUN and pending_mixes:
    pending_mixes = pending_mixes[:1]
    print(f'\nDRY_RUN: processing only 1 mix')

print(f'\n{"="*70}')
print(f'DATASET SUMMARY')
print(f'{"="*70}')
print(f'Total mixes available: {len(all_mixes)}')
print(f'Step 7 already done: {len(step7_done)}')
print(f'Ready for processing: {len(pending_mixes)}')
print(f'Total transitions across all: {sum(m["n_transitions"] for m in all_mixes)}')
print(f'Total residuals to extract: {sum(m["n_transitions"] for m in pending_mixes)}')

In [ ]:
import sys
if '/kaggle/working/ai-dj/v4' not in sys.path:
    sys.path.insert(0, '/kaggle/working/ai-dj/v4')

import logging
import time
import shutil
import tempfile
import numpy as np
from pathlib import Path
from multiprocessing import Pool
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("step7")

from aidj import config
from aidj.curves.stem_curve_extractor import StemCurveExtractor
from aidj.stems.stem_cache import StemCache

# -------------------------------------------------------
# HF download helpers
# -------------------------------------------------------

def download_track_stems(tid):
    """Download all 4 stem OGGs for a track from HF."""
    stem_dir = DATA_ROOT_PATH / "stems" / "tracks" / tid
    ext = config.STEM_EXT[config.STEM_FORMAT]
    all_ok = True
    for stem in config.STEMS:
        local = stem_dir / f"{stem}{ext}"
        if local.exists():
            continue
        stem_dir.mkdir(parents=True, exist_ok=True)
        try:
            hf_hub_download(
                repo_id=HF_REPO,
                filename=f"stems/tracks/{tid}/{stem}{ext}",
                repo_type="dataset", token=HF_TOKEN,
                local_dir=DATA_ROOT,
            )
        except Exception as e:
            log.warning(f"  Could not download stem {stem} for track {tid}: {e}")
            all_ok = False
    return all_ok

def download_mix_seg_stems(tran_id):
    """Download all 4 stem OGGs for a mix segment from HF."""
    seg_dir = DATA_ROOT_PATH / "stems" / "mix_segments" / tran_id
    ext = config.STEM_EXT[config.STEM_FORMAT]
    all_ok = True
    for stem in config.STEMS:
        local = seg_dir / f"{stem}{ext}"
        if local.exists():
            continue
        seg_dir.mkdir(parents=True, exist_ok=True)
        try:
            hf_hub_download(
                repo_id=HF_REPO,
                filename=f"stems/mix_segments/{tran_id}/{stem}{ext}",
                repo_type="dataset", token=HF_TOKEN,
                local_dir=DATA_ROOT,
            )
        except Exception as e:
            log.warning(f"  Could not download stem {stem} for segment {tran_id}: {e}")
            all_ok = False
    return all_ok

# -------------------------------------------------------
# Worker: extract curves for one transition
# -------------------------------------------------------

def _extract_one(args):
    tran, data_root_str, output_path_str = args
    tran_id = tran["tran_id"]
    prev_id = tran["track_id_prev"]
    next_id = tran["track_id_next"]

    data_root = Path(data_root_str)
    output_path = Path(output_path_str)

    if output_path.exists():
        return tran_id, True, "cached"

    prev_cue_secs = tran.get("track_cue_in_time_prev", 0)
    next_cue_secs = tran.get("track_cue_in_time_next", 0)
    MAX_CUE_SECS = 3600
    if prev_cue_secs > MAX_CUE_SECS or next_cue_secs > MAX_CUE_SECS:
        return tran_id, False, (f"bogus cue times")

    stem_cache = StemCache(data_root)
    mix_seg_stems = stem_cache.load_stems("mix_segments", tran_id)
    prev_stems = stem_cache.load_stems("tracks", prev_id)
    next_stems = stem_cache.load_stems("tracks", next_id)

    if mix_seg_stems is None or prev_stems is None or next_stems is None:
        missing = []
        if mix_seg_stems is None: missing.append(f"mix_seg/{tran_id}")
        if prev_stems is None: missing.append(f"track/{prev_id}")
        if next_stems is None: missing.append(f"track/{next_id}")
        return tran_id, False, f"missing stems"

    try:
        region_len = min(len(v) for v in mix_seg_stems.values())
        max_samples = int(config.MAX_TRANSITION_SECS * config.OPT_SR)
        region_len = min(region_len, max_samples)

        if region_len < config.OPT_SR:
            return tran_id, False, "too short"

        mix_region = {s: mix_seg_stems[s][:region_len] for s in config.STEMS}
        prev_start = int(prev_cue_secs * config.OPT_SR)
        next_start = int(next_cue_secs * config.OPT_SR)

        prev_track_len = min(len(v) for v in prev_stems.values())
        next_track_len = min(len(v) for v in next_stems.values())
        if prev_start >= prev_track_len or next_start >= next_track_len:
            return tran_id, False, "cue out of range"

        prev_region = {s: prev_stems[s][prev_start:prev_start + region_len] for s in config.STEMS}
        next_region = {s: next_stems[s][next_start:next_start + region_len] for s in config.STEMS}

        min_len = min(
            region_len,
            min(len(v) for v in prev_region.values()),
            min(len(v) for v in next_region.values()),
        )
        if min_len < config.OPT_SR:
            return tran_id, False, "too short after trim"

        mix_region = {s: v[:min_len] for s, v in mix_region.items()}
        prev_region = {s: v[:min_len] for s, v in prev_region.items()}
        next_region = {s: v[:min_len] for s, v in next_region.items()}

        extractor = StemCurveExtractor()
        curves = extractor.extract_transition_curves(mix_region, prev_region, next_region)

        if curves is not None:
            extractor.save_curves(curves, output_path)
            return tran_id, True, None
        else:
            return tran_id, False, "optimization failed"

    except Exception as e:
        return tran_id, False, str(e)[:50]

# -------------------------------------------------------
# Per-mix processing
# -------------------------------------------------------

def process_mix(mix):
    """Download stems, extract curves, return (n_ok, n_total)."""
    mix_id = mix["id"]
    transitions = mix["transitions"]
    data_root = DATA_ROOT_PATH

    track_ids = set()
    for t in transitions:
        if t.get("track_id_prev"): track_ids.add(t["track_id_prev"])
        if t.get("track_id_next"): track_ids.add(t["track_id_next"])

    log.info(f"  Downloading stems for {len(track_ids)} tracks + {len(transitions)} segments...")
    with ThreadPoolExecutor(max_workers=8) as tp:
        list(tp.map(download_track_stems, sorted(track_ids)))
        list(tp.map(lambda t: download_mix_seg_stems(t["tran_id"]), transitions))

    curves_dir = data_root / "results" / "stem_curves"
    mix_curves_dir = curves_dir / mix_id
    mix_curves_dir.mkdir(parents=True, exist_ok=True)

    work_items = []
    already_done = 0
    for tran in transitions:
        tran_id = tran["tran_id"]
        out = mix_curves_dir / f"{tran_id}.npz"
        if out.exists():
            already_done += 1
        else:
            work_items.append((tran, str(data_root), str(out)))

    log.info(f"  Curves: {len(work_items)} pending, {already_done} cached")

    n_ok = already_done
    n_fail = 0
    total = len(transitions)
    WORKER_TIMEOUT = 90

    if work_items:
        remaining = list(enumerate(work_items))
        while remaining:
            with Pool(min(CURVE_WORKERS, len(remaining))) as pool:
                futures = []
                for idx, item in remaining:
                    futures.append((idx, item[0]["tran_id"], pool.apply_async(_extract_one, (item,))))

                new_remaining = []
                pool_killed = False
                for idx, tran_id, future in futures:
                    if pool_killed:
                        new_remaining.append((idx, work_items[idx]))
                        continue
                    try:
                        _, ok, err = future.get(timeout=WORKER_TIMEOUT)
                        if ok:
                            n_ok += 1
                            log.info(f"  [{n_ok}/{total}] {tran_id}: OK")
                        else:
                            n_fail += 1
                            log.warning(f"  {tran_id}: {err}")
                    except Exception:
                        n_fail += 1
                        log.warning(f"  {tran_id}: timeout ({WORKER_TIMEOUT}s)")
                        pool.terminate()
                        pool.join()
                        pool_killed = True
            remaining = new_remaining

    log.info(f"  Curves: {n_ok} ok, {n_fail} failed, {total} total")
    return n_ok, total

def upload_curves(mix_id):
    """Upload curve files for one mix to HF."""
    curves_dir = DATA_ROOT_PATH / "results" / "stem_curves"
    mix_curves_dir = curves_dir / mix_id
    curve_files = list(mix_curves_dir.glob("*.npz"))
    if not curve_files:
        log.warning(f"  No curve files for {mix_id}")
        return 0

    log.info(f"  Uploading {len(curve_files)} curves...")
    with tempfile.TemporaryDirectory() as staging_dir:
        staging = Path(staging_dir)
        for f in curve_files:
            dst = staging / "results" / "stem_curves" / mix_id / f.name
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dst)
        api.upload_large_folder(
            folder_path=str(staging),
            repo_id=HF_REPO,
            repo_type="dataset",
        )
    return len(curve_files)

def cleanup_mix_stems(mix_id, transitions, remaining_track_ids):
    """Delete local stem files to free disk."""
    deleted = 0
    data_root = DATA_ROOT_PATH

    for tran in transitions:
        seg_dir = data_root / "stems" / "mix_segments" / tran["tran_id"]
        if seg_dir.exists():
            shutil.rmtree(seg_dir)
            deleted += 1

    track_ids = set()
    for t in transitions:
        if t.get("track_id_prev"): track_ids.add(t["track_id_prev"])
        if t.get("track_id_next"): track_ids.add(t["track_id_next"])

    for tid in track_ids:
        if tid not in remaining_track_ids:
            stem_dir = data_root / "stems" / "tracks" / tid
            if stem_dir.exists():
                shutil.rmtree(stem_dir)
                deleted += 1

# -------------------------------------------------------
# Main loop
# -------------------------------------------------------

if not pending_mixes:
    log.info('All mixes already have curves!')
else:
    session_start = time.time()

    for mix_idx, mix in enumerate(pending_mixes):
        mix_id = mix["id"]
        transitions = mix["transitions"]
        mix_start = time.time()

        log.info(f'\n{"="*60}')
        log.info(f'[{mix_idx+1}/{len(pending_mixes)}] {mix_id} ({len(transitions)} transitions)')
        log.info(f'{"="*60}')

        remaining_mixes = pending_mixes[mix_idx + 1:]
        remaining_track_ids = set()
        for m in remaining_mixes:
            for t in m["transitions"]:
                if t.get("track_id_prev"): remaining_track_ids.add(t["track_id_prev"])
                if t.get("track_id_next"): remaining_track_ids.add(t["track_id_next"])

        try:
            n_ok, n_total = process_mix(mix)
        except Exception as e:
            log.error(f"Process failed: {e}")
            continue

        try:
            upload_curves(mix_id)
        except Exception as e:
            log.error(f"Upload failed: {e}")
            continue

        if n_ok > 0:
            progress["curves"].append(mix_id)
            push_progress(progress)
        else:
            log.warning(f"0 curves for {mix_id}, skipping")

        cleanup_mix_stems(mix_id, transitions, remaining_track_ids)

        elapsed = (time.time() - mix_start) / 60
        total_elapsed = (time.time() - session_start) / 3600
        mixes_left = len(pending_mixes) - (mix_idx + 1)
        if mix_idx > 0:
            avg_per_mix = (time.time() - session_start) / (mix_idx + 1) / 60
            eta_min = mixes_left * avg_per_mix
            log.info(f'Done in {elapsed:.1f}m | session: {total_elapsed:.2f}h | ETA: {eta_min:.0f}m ({mixes_left} left)')

    log.info(f'\n{"="*60}')
    log.info(f'Step 7 complete! Curves: {len(progress["curves"])} mixes')

In [ ]:
# Verify: reload all progress and show final status
all_hf_files = list(list_repo_files(repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN))
prog_files = [f for f in all_hf_files if f.startswith("phase_b_progress_step7_") and f.endswith(".json")]
all_step7_done = set()
for pf in prog_files:
    try:
        local = hf_hub_download(
            repo_id=HF_REPO, filename=pf,
            repo_type="dataset", token=HF_TOKEN,
            local_dir=DATA_ROOT, force_download=True,
        )
        with open(local) as f:
            p = json.load(f)
        all_step7_done.update(p.get("curves", []))
    except Exception as e:
        print(f'Warning: {pf}: {e}')

# Count curve files on HF
curve_files = [f for f in all_hf_files if f.startswith("results/stem_curves/") and f.endswith(".npz")]
curves_by_mix = defaultdict(int)
for f in curve_files:
    parts = f.split("/")
    if len(parts) >= 3:
        curves_by_mix[parts[2]] += 1

print(f'\n{"="*70}')
print(f'FINAL STEP 7 STATUS')
print(f'{"="*70}')
print(f'Mixes with curves on HF: {len(curves_by_mix)}')
print(f'Total curve files: {len(curve_files)}')
print(f'\nMixes with curves:')
for mix_id in sorted(curves_by_mix):
    print(f'  {mix_id}: {curves_by_mix[mix_id]} curves')
print(f'\n✅ Ready for Step 8 (training)!')